# Almacenamiento de archivos en Databricks

En Azure Databricks hay varios conceptos para almacenar o acceder a archivos, algunos son mecanismos de almacenamiento y otros son simplemente formas de acceso.

## 1. Unity Catalog Volumes (recomendado actualmente)

Los Volumes son la forma moderna de almacenar archivos no estructurados (CSV, JSON, imágenes, modelos, etc.) dentro del gobierno de datos de Unity Catalog.

Por ejemplo: `dbfs:/dbfs/Volumes/catalog/schema/volume_name/`

**Nota:** En Databricks, Schema = Database.

## 2. Cloud Storage directamente (ADLS Gen2)

Los archivos realmente terminan viviendo en un storage como:

- Azure Data Lake Storage Gen2 (ADLS)
- Blob Storage
- Amazon S3 (AWS)
- Google Cloud Storage (GCP)

Por ejemplo: `abfss://raw@midatalake.dfs.core.windows.net/sales/`

## 3. Mount Points (legacy)

Antes de Unity Catalog era común usarlos.

Por ejemplo: `dbfs:/mnt/unilever/PBL/OTC002/`

In [0]:
# Show all mount points
# display(dbutils.fs.mounts())

In [0]:
# display(dbutils.fs.ls("dbfs:/mnt/unilever/PBL/OTC0023"))

## 4. DBFS Root

Databricks también posee un área propia llamada DBFS (Databricks File System).

Se usa para archivos temporales, Logs, cargas manuales.

Por ejemplo: `dbfs:/FileStore/`

**Nota:** No suele utilizarse para almacenamiento corporativo de datos productivos.

## Sistema de archivos en Databricks

Con los siguientes comandos, se lista el contenido del directorio raíz del sistema de archivos.

In [0]:
%fs
ls /

path,name,size,modificationTime
dbfs:/FileStore/,FileStore/,0,1732128488000
dbfs:/Volume/,Volume/,0,0
dbfs:/Volumes/,Volumes/,0,0
dbfs:/databricks-datasets/,databricks-datasets/,0,0
dbfs:/databricks-results/,databricks-results/,0,0
dbfs:/dbfs/,dbfs/,0,1732297723000
dbfs:/mnt/,mnt/,0,1716390364000
dbfs:/tmp/,tmp/,0,1726169880000
dbfs:/user/,user/,0,1718202675000
dbfs:/volume/,volume/,0,0


O equivalentemente,

In [0]:
display(dbutils.fs.ls("/"))

path,name,size,modificationTime
dbfs:/FileStore/,FileStore/,0,1732128488000
dbfs:/Volume/,Volume/,0,0
dbfs:/Volumes/,Volumes/,0,0
dbfs:/databricks-datasets/,databricks-datasets/,0,0
dbfs:/databricks-results/,databricks-results/,0,0
dbfs:/dbfs/,dbfs/,0,1732297723000
dbfs:/mnt/,mnt/,0,1716390364000
dbfs:/tmp/,tmp/,0,1726169880000
dbfs:/user/,user/,0,1718202675000
dbfs:/volume/,volume/,0,0


Unity Catalog no reemplaza al Data Lake. Tus archivos siguen estando en ADLS Gen2; Unity Catalog agrega la capa de gobernanza, seguridad y catálogo, y los Volumes son la forma moderna de exponer esos archivos dentro de Databricks.

## Catalogs en Unity Catalog

Si al ejecutar `SHOW CATALOGS;`, solo devuelve: `spark_catalog`, entonces puede que Unity Catalog (UC) aun no esté habilitado y se esté usando el catalogo legacy `Hive Metastore (spark_catalog)`.

In [0]:
%sql
SHOW CATALOGS;

catalog
spark_catalog


In [0]:
%sql
SELECT current_catalog()

current_catalog()
spark_catalog


Veamos los schemas (o databases) en el catalogo `spark_catalog` (equivalente a `hive_metastore`):

In [0]:
%sql
SHOW SCHEMAS;

databaseName
bat_agentes_data
bat_ai_demo
bat_controllers
bat_scintilla_catman
bat_scintilla_supplier
bat_sharemkt
bat_sharemkt_weekly
br_unilever_cpfr_2
br_unilever_logistics
br_unilever_sky


Usaremos el esquema `dwh_mx_unilever_status_one` que Rodrigo usa.

In [0]:
%sql
USE SCHEMA dwh_mx_unilever_status_one;

In [0]:
%sql
SHOW TABLES IN dwh_mx_unilever_status_one;


database,tableName,isTemporary
dwh_mx_unilever_status_one,7tbl,false
dwh_mx_unilever_status_one,almacen,false
dwh_mx_unilever_status_one,api_rc,false
dwh_mx_unilever_status_one,api_yms_ib,false
dwh_mx_unilever_status_one,api_yms_ob,false
dwh_mx_unilever_status_one,bolo,false
dwh_mx_unilever_status_one,cargas_export,false
dwh_mx_unilever_status_one,cuota,false
dwh_mx_unilever_status_one,curvas_facturacion_2026,false
dwh_mx_unilever_status_one,curvas_facturacion_2026_v8,false


## Upload table to Unity Catalog manually
![table in UC.png](/Workspace/Users/antonio.arteaga@unilever.com/pyspark/PySpark-Pandas/images/table in UC.png)